# Particle Tracking Velocimetry of rod-like particles

Tracking pipeline for the paper *A deep learning-enhanced PTV framework for
simultaneous translational and rotational tracking of rod-like particles in
non-Newtonian fluids*.

The notebook takes the raw image sequences of one experiment, segments every
fiber with the fine-tuned YOLOv11 model in `model/best.pt`, links the detections
across frames with an alpha-beta-gamma predictive filter, and writes one JSON
trajectory file per experiment into `results/`.

**Running it requires the raw image sequences**, which are not distributed with
this repository because of their size. The trajectory files they produce are
included, so every downstream analysis can be reproduced without re-running the
segmentation. See the README for details.

Pipeline: **segmentation -> characterisation -> association -> post-processing**.


## 1. Configuration

Every parameter used by the pipeline is set here.


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Every path below is derived from the repository root, so this notebook runs
# unchanged on Linux, macOS and Windows.

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MODEL_PATH  = REPO_ROOT / "model" / "best.pt"      # fine-tuned YOLOv11 weights
RESULTS_DIR = REPO_ROOT / "results"                # trajectory JSON output
RUNS_DIR    = REPO_ROOT / "runs" / "segment"       # annotated frames output

# Root folder holding the raw image sequences, one sub-folder per experiment:
#     <IMAGES_ROOT>/<N> Fibras/Cam 1/*.bmp
# The image sequences are ~27 GB and are NOT distributed with this repository;
# see the README for how to obtain them.
IMAGES_ROOT = REPO_ROOT / "images"

# --- Image acquisition -------------------------------------------------------
FPS      = 200            # camera frame rate [Hz]
DELTA_T  = 1.0 / FPS      # time between consecutive frames [s]
N_FRAMES = 600            # frames processed per experiment (3 s at 200 fps)

# --- alpha-beta-gamma filter gains -------------------------------------------
# High position and velocity gains keep the predictor responsive; the low
# acceleration gain suppresses the frame-to-frame jitter of the segmentation.
ALPHA = 0.95              # position gain
BETA  = 0.95              # velocity gain
GAMMA = 0.05              # acceleration gain

# --- Association gate --------------------------------------------------------
# Largest change accepted between consecutive frames for two detections to be
# considered the same fiber.
MAX_DELTA_X     = 10      # [px]
MAX_DELTA_Y     = 10      # [px]
MAX_DELTA_ANGLE = 5       # [deg]

# --- Segmentation ------------------------------------------------------------
CONF_THRESHOLD = 0.25     # YOLO confidence threshold

# Ultralytics caps the number of detections returned per image. The published
# results were produced with the library default of 300. That cap saturates the
# 800-fiber case, where exactly 300 detections are returned in 564 of its 600
# frames, so the value is set explicitly here to keep the behaviour visible and
# reproducible. Raise it to re-run that case without the cap.
MAX_DETECTIONS = 300

# --- Post-processing ---------------------------------------------------------
# Tracks shorter than this are discarded. They are dominated by transient
# artefacts, mainly elongated bubbles misclassified as fibers.
MIN_TRACK_FRAMES = 20     # 0.1 s at 200 fps

# --- Experiments -------------------------------------------------------------
FIBER_COUNTS = ["25", "50", "100", "200", "400", "800"]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {REPO_ROOT}")
print(f"Model weights   : {MODEL_PATH}  (exists: {MODEL_PATH.exists()})")
print(f"Image sequences : {IMAGES_ROOT}  (exists: {IMAGES_ROOT.exists()})")

## 2. Imports


In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import json                      # read and write the trajectory files
import math                      # scalar maths

import cv2                       # image I/O and annotation
import numpy as np               # numerical arrays
import matplotlib.pyplot as plt  # diagnostic plots
from ultralytics import YOLO     # YOLOv11 instance segmentation

## 3. Alpha-beta-gamma predictive filter

A reduced-order Kalman filter with fixed gains. It predicts the position and
orientation of each fiber in the next frame; that prediction defines the search
gate used to decide whether a new detection continues an existing track.


In [ ]:
# =============================================================================
# ALPHA-BETA-GAMMA PREDICTIVE FILTER
# =============================================================================
# The filter predicts where each fiber should appear in the next frame, and the
# prediction defines the search gate used to associate detections across frames.
# Two independent chains are maintained: one for the centroid (x, y) and one for
# the orientation angle.
#
# State layout (list of lists, kept as in the original implementation):
#     [0] [xx, xy]                position           [px]
#     [1] [vx, vy]                velocity           [px/s]
#     [2] [ax, ay]                acceleration       [px/s^2]
#     [3] [angle]                 orientation        [deg]
#     [4] [omega]                 angular velocity   [deg/s]
#     [5] [angular_acceleration]  angular accel.     [deg/s^2]
#     [6] [length]                projected length   [px]
#
# Measurement layout:
#     [0] [zx, zy]    measured centroid    [px]
#     [1] [z_angle]   measured orientation [deg]
#     [2] [z_length]  measured length      [px]


# -----------------------------------------------------------------------------
# 1) Angle helpers
# -----------------------------------------------------------------------------
def normalize_angle_difference(measured_angle, filtered_angle):
    """Smallest signed difference between two angles, wrapped to (-180, 180].

    Prevents spurious jumps such as 179 -> -179 from being read as a 358 degree
    rotation.
    """
    difference = measured_angle - filtered_angle
    while difference > 180:
        difference -= 360
    while difference <= -180:
        difference += 360
    return difference


def normalize_angle(angle):
    """Wrap an angle to the (-180, 180] range."""
    while angle > 180:
        angle -= 360
    while angle <= -180:
        angle += 360
    return angle


# -----------------------------------------------------------------------------
# 2) Generic alpha, beta and gamma updates (linear position)
# -----------------------------------------------------------------------------
def alpha_update(value, previous_value, measurement, dt):
    """Correct the position estimate with the measurement residual."""
    return value + ALPHA * (measurement - previous_value)


def beta_update(value, previous_value, measurement, dt):
    """Correct the velocity estimate with the measurement residual."""
    return value + BETA * ((measurement - previous_value) / dt)


def gamma_update(value, previous_value, measurement, dt):
    """Correct the acceleration estimate with the measurement residual."""
    return value + GAMMA * ((measurement - previous_value) / (dt ** 2) * 2)


# -----------------------------------------------------------------------------
# 3) Angle-specific alpha, beta and gamma updates
# -----------------------------------------------------------------------------
def alpha_update_angle(filtered_angle, measured_angle):
    """Correct the orientation estimate, wrapping the residual first."""
    difference = normalize_angle_difference(measured_angle, filtered_angle)
    return normalize_angle(filtered_angle + ALPHA * difference)


def beta_update_angle(filtered_omega, filtered_angle, measured_angle, dt):
    """Correct the angular velocity estimate."""
    difference = normalize_angle_difference(measured_angle, filtered_angle)
    return filtered_omega + BETA * (difference / dt)


def gamma_update_angle(filtered_angular_acc, filtered_angle, measured_angle, dt):
    """Correct the angular acceleration estimate."""
    difference = normalize_angle_difference(measured_angle, filtered_angle)
    return filtered_angular_acc + GAMMA * (difference / (dt ** 2) * 0.5)


# -----------------------------------------------------------------------------
# 4) One filter step: correction followed by prediction
# -----------------------------------------------------------------------------
def abg_filter(state, measurement, frame_step=1):
    """Advance the filter by one step and return the predicted state.

    Args:
        state:       current state, in the layout documented above.
        measurement: current detection, in the measurement layout above.
        frame_step:  number of frames between the state and the measurement.

    Returns:
        The predicted state for the next frame.
    """
    dt = DELTA_T * frame_step

    # --- Unpack the current state ---
    xx_prev, xy_prev = state[0]
    vx_prev, vy_prev = state[1]
    ax_prev, ay_prev = state[2]
    angle_prev = state[3][0]
    omega_prev = state[4][0]
    angular_acc_prev = state[5][0]

    # --- Unpack the measurement ---
    zx, zy = measurement[0]
    z_angle = measurement[1][0]
    z_length = measurement[2][0]

    # --- Correction ---
    xx_corr = alpha_update(xx_prev, xx_prev, zx, dt)
    xy_corr = alpha_update(xy_prev, xy_prev, zy, dt)
    vx_corr = beta_update(vx_prev, xx_prev, zx, dt)
    vy_corr = beta_update(vy_prev, xy_prev, zy, dt)
    ax_corr = gamma_update(ax_prev, xx_prev, zx, dt)
    ay_corr = gamma_update(ay_prev, xy_prev, zy, dt)

    angle_corr = alpha_update_angle(angle_prev, z_angle)
    omega_corr = beta_update_angle(omega_prev, angle_prev, z_angle, dt)
    angular_acc_corr = gamma_update_angle(angular_acc_prev, angle_prev, z_angle, dt)

    # --- Prediction for the next frame ---
    xx_pred = xx_corr + vx_corr * dt + 0.5 * ax_corr * (dt ** 2)
    xy_pred = xy_corr + vy_corr * dt + 0.5 * ay_corr * (dt ** 2)
    vx_pred = vx_corr + ax_corr * dt
    vy_pred = vy_corr + ay_corr * dt
    angle_pred = normalize_angle(
        angle_corr + omega_corr * dt + 0.5 * angular_acc_corr * (dt ** 2)
    )
    omega_pred = omega_corr + angular_acc_corr * dt

    return [
        [xx_pred, xy_pred],
        [vx_pred, vy_pred],
        [ax_corr, ay_corr],
        [angle_pred],
        [omega_pred],
        [angular_acc_corr],
        [z_length],           # the length is carried over, not filtered
    ]


# -----------------------------------------------------------------------------
# 5) Initial guess for a newly detected fiber
# -----------------------------------------------------------------------------
def initial_guess(measurement):
    """Build the first state of a track, with zero velocity and acceleration.

    Args:
        measurement: [[x, y], [angle], [length]] of the first detection.
    """
    x, y = measurement[0]
    angle = measurement[1][0]
    length = measurement[2][0]

    vx = vy = ax = ay = 0.0
    omega = angular_acc = 0.0

    x_pred = x + vx * DELTA_T + 0.5 * ax * (DELTA_T ** 2)
    y_pred = y + vy * DELTA_T + 0.5 * ay * (DELTA_T ** 2)
    angle_pred = normalize_angle(
        angle + omega * DELTA_T + 0.5 * angular_acc * (DELTA_T ** 2)
    )

    return [
        [x_pred, y_pred],
        [vx, vy],
        [ax, ay],
        [angle_pred],
        [omega],
        [angular_acc],
        [length],
    ]

## 4. Segmentation and frame annotation

Each fiber is reduced to three descriptors taken from its bounding box: the
centroid, the longer side of the box as the projected length, and the angle of
the box diagonal as the orientation.


In [ ]:
# =============================================================================
# SEGMENTATION AND FRAME ANNOTATION
# =============================================================================

def to_json_serialisable(obj):
    """Convert NumPy scalars and arrays into plain Python types for json.dump."""
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Unsupported type for JSON serialisation: {type(obj)}")


def load_model(image_folder, output_folder):
    """Load the fine-tuned weights and list the frames of one experiment.

    Args:
        image_folder:  folder holding the raw frames of a single experiment.
        output_folder: folder where the annotated frames will be written.

    Returns:
        (model, frame_paths) with the frames sorted by file name and truncated
        to N_FRAMES.
    """
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model weights not found: {MODEL_PATH}")
    if not image_folder.is_dir():
        raise FileNotFoundError(f"Image folder not found: {image_folder}")

    output_folder.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(MODEL_PATH))

    frame_paths = sorted(
        p for p in image_folder.iterdir()
        if p.suffix.lower() in (".bmp", ".jpg", ".jpeg", ".png")
    )[:N_FRAMES]

    if not frame_paths:
        raise FileNotFoundError(f"No images found in {image_folder}")

    print(f"  {len(frame_paths)} frames found, annotated output -> {output_folder}")
    return model, frame_paths


def segment_frame(model, frame_path, output_folder):
    """Run instance segmentation on one frame and describe every detection.

    Each fiber is reduced to three descriptors taken from its bounding box:
      * centroid  = box centre
      * length    = longer side of the box
      * angle     = atan2(box height, box width)

    NOTE ON THE ANGLE. Both box sides are positive, so the angle is folded into
    [0, 90] degrees: it measures the inclination with respect to the horizontal
    and does not distinguish +30 from -30 degrees. This is the convention used
    throughout the paper, and every reported orientation statistic follows it.

    Returns:
        (centroids, angles, lengths, scores, boxes), or five None values when
        the frame contains no detection.
    """
    results = model.predict(
        source=str(frame_path),
        conf=CONF_THRESHOLD,
        max_det=MAX_DETECTIONS,
        save=True,
        project=str(output_folder.parent),
        name=output_folder.name,
        exist_ok=True,
        show_labels=False,
        line_width=1,
        verbose=False,
    )

    boxes_obj = results[0].boxes
    if boxes_obj is None or len(boxes_obj) == 0:
        print(f"  No objects detected in {frame_path.name}")
        return None, None, None, None, None

    boxes = boxes_obj.xyxy.cpu().numpy()
    scores = boxes_obj.conf.cpu().numpy()

    centroids, lengths, angles = [], [], []
    for x1, y1, x2, y2 in boxes:
        width = x2 - x1
        height = y2 - y1
        centroids.append(((x1 + x2) / 2.0, (y1 + y2) / 2.0))
        lengths.append(max(width, height))
        angles.append(np.degrees(np.arctan2(height, width)))

    return centroids, angles, lengths, scores, boxes


def annotate_frame(output_folder, frame_path, fiber_ids_this_frame, boxes):
    """Overlay the track identifier of every detection on the annotated frame.

    Ultralytics has already written the segmentation overlay to disk; this adds
    the track id next to each box and a reference rectangle whose size equals
    the association gate (MAX_DELTA_X by MAX_DELTA_Y).
    """
    # Depending on the Ultralytics version the overlay is written either with
    # the original extension or as JPEG, so both are checked.
    image_path = output_folder / frame_path.name
    if not image_path.exists():
        image_path = output_folder / f"{frame_path.stem}.jpg"
    if not image_path.exists():
        print(f"  [WARNING] Annotated frame not found for {frame_path.name}")
        return

    image = cv2.imread(str(image_path))
    if image is None:
        print(f"  [WARNING] Could not read: {image_path}")
        return

    for detection_index, track_id in fiber_ids_this_frame.items():
        x1, y1, _, _ = boxes[detection_index]
        cv2.putText(
            image,
            track_id,
            (int(x1), int(y1) - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),        # BGR: blue
            2,
        )

    height, width = image.shape[:2]
    cv2.rectangle(
        image,
        (width - MAX_DELTA_X - 10, 10),
        (width - 10, 10 + MAX_DELTA_Y),
        (0, 255, 0),            # BGR: green
        2,
    )

    cv2.imwrite(str(image_path), image)

## 5. Tracking

Detections are linked to the tracks seen in the immediately preceding frame.
The first previous track whose predicted state falls inside the gate wins the
detection; unmatched detections start a new track.


In [ ]:
# =============================================================================
# TRACKING: ASSOCIATE DETECTIONS ACROSS CONSECUTIVE FRAMES
# =============================================================================

def track_experiment(fiber_count):
    """Segment and track one experiment, and write its trajectory file.

    For every frame, each detection is compared against the tracks seen in the
    previous frame. The alpha-beta-gamma filter supplies the predicted state and
    the comparison is accepted when the position and angle residuals fall inside
    the gate. Unmatched detections start a new track.

    Association is greedy: the first previous track that passes the gate wins.
    Tracks are only carried over from the immediately preceding frame, so a
    single missed detection ends a track; this is the source of the trajectory
    fragmentation quantified in the paper.

    Args:
        fiber_count: experiment label, e.g. "200".

    Returns:
        Path of the JSON file written.
    """
    # "Fibras" and "Cam 1" are the folder names produced by the acquisition
    # software; adjust them here if your capture is laid out differently.
    image_folder = IMAGES_ROOT / f"{fiber_count} Fibras" / "Cam 1"
    output_folder = RUNS_DIR / fiber_count

    print(f"Tracking the {fiber_count}-fiber experiment")
    model, frame_paths = load_model(image_folder, output_folder)

    tracks = {}                    # track id -> per-frame measurements
    next_track_id = 0              # running identifier
    ids_current_frame = []         # track ids present in the current frame
    detections_per_frame = []      # detection count, one entry per frame

    for frame_index, frame_path in enumerate(frame_paths):
        centroids, angles, lengths, scores, boxes = segment_frame(
            model, frame_path, output_folder
        )

        # Carry the identifiers of the previous frame over before resetting.
        ids_previous_frame = ids_current_frame
        ids_current_frame = []

        if centroids is None:
            detections_per_frame.append(0)
            continue
        detections_per_frame.append(len(scores))

        # Maps the index of a detection in this frame to its track id.
        fiber_ids_this_frame = {}

        for i in range(len(scores)):
            measurement = [[centroids[i][0], centroids[i][1]],
                           [angles[i]],
                           [lengths[i]]]
            matched = False

            # --- Try to continue one of the tracks seen in the previous frame ---
            for track_id in ids_previous_frame:
                if track_id in ids_current_frame:
                    continue        # already claimed by another detection

                prediction = abg_filter(tracks[track_id]["kalman"][-1],
                                        measurement, frame_step=1)

                delta_x = abs(prediction[0][0] - centroids[i][0])
                delta_y = abs(prediction[0][1] - centroids[i][1])
                if delta_x >= MAX_DELTA_X or delta_y >= MAX_DELTA_Y:
                    continue

                delta_angle = abs(prediction[3][0] - angles[i])
                if delta_angle >= MAX_DELTA_ANGLE:
                    continue

                tracks[track_id]["centroide"].append(
                    [centroids[i][0], centroids[i][1]])
                tracks[track_id]["largo_maximo"].append([lengths[i]])
                tracks[track_id]["angulo"].append([angles[i]])
                tracks[track_id]["frame"].append([frame_index + 1])
                tracks[track_id]["kalman"].append(prediction)

                fiber_ids_this_frame[i] = track_id
                ids_current_frame.append(track_id)
                matched = True
                break

            # --- Otherwise start a new track ---
            if not matched:
                next_track_id += 1
                track_id = str(next_track_id)
                tracks[track_id] = {
                    "centroide":    [[centroids[i][0], centroids[i][1]]],
                    "largo_maximo": [[lengths[i]]],
                    "angulo":       [[angles[i]]],
                    "frame":        [[frame_index + 1]],
                    "kalman":       [initial_guess(measurement)],
                }
                fiber_ids_this_frame[i] = track_id
                ids_current_frame.append(track_id)

        annotate_frame(output_folder, frame_path, fiber_ids_this_frame, boxes)

    tracks["fibras_por_frame"] = detections_per_frame

    output_path = RESULTS_DIR / f"trajectories_{fiber_count}_raw.json"
    with open(output_path, "w", encoding="utf-8") as handle:
        json.dump(tracks, handle, default=to_json_serialisable)

    n_tracks = sum(1 for key in tracks if key.isdigit())
    print(f"  {n_tracks} tracks written to {output_path.name}")
    return output_path

## 6. Post-processing

Tracks shorter than `MIN_TRACK_FRAMES` are discarded. Both the raw and the
filtered trajectory files are kept, so the effect of this step can be audited.


In [ ]:
# =============================================================================
# POST-PROCESSING: DROP SHORT TRACKS
# =============================================================================

def filter_short_tracks(fiber_count, min_frames=None):
    """Keep only the tracks that span at least `min_frames` frames.

    Short tracks are dominated by transient artefacts, mainly elongated bubbles
    that the segmentation model misclassifies as fibers, and by spurious
    associations in crowded regions.

    Args:
        fiber_count: experiment label, e.g. "200".
        min_frames:  minimum track length; defaults to MIN_TRACK_FRAMES.

    Returns:
        Path of the JSON file written.
    """
    if min_frames is None:
        min_frames = MIN_TRACK_FRAMES

    input_path = RESULTS_DIR / f"trajectories_{fiber_count}_raw.json"
    output_path = RESULTS_DIR / f"trajectories_{fiber_count}_filtered.json"

    with open(input_path, "r", encoding="utf-8") as handle:
        raw = json.load(handle)

    filtered = {}
    if "fibras_por_frame" in raw:
        filtered["fibras_por_frame"] = raw["fibras_por_frame"]

    for track_id, track in raw.items():
        if track_id == "fibras_por_frame":
            continue
        if len(track.get("frame", [])) >= min_frames:
            filtered[track_id] = track

    with open(output_path, "w", encoding="utf-8") as handle:
        json.dump(filtered, handle, ensure_ascii=False)

    kept = sum(1 for key in filtered if key.isdigit())
    total = sum(1 for key in raw if key.isdigit())
    print(f"  {kept} of {total} tracks kept (>= {min_frames} frames) "
          f"-> {output_path.name}")
    return output_path

## 7. Run the pipeline


In [ ]:
# =============================================================================
# RUN THE FULL PIPELINE
# =============================================================================
# Requires the raw image sequences under IMAGES_ROOT. Expect a few hours per
# experiment on a single GPU; the 800-fiber case is by far the slowest.

for fiber_count in FIBER_COUNTS:
    track_experiment(fiber_count)
    filter_short_tracks(fiber_count)

print("\nDone. Trajectory files are in", RESULTS_DIR)

## 8. Diagnostics

The two cells below read the trajectory files back and reproduce the basic
checks used while developing the pipeline. They run on the files shipped in
`results/`, so they work without the raw images.


In [ ]:
# =============================================================================
# DIAGNOSTIC 1: CUMULATIVE NUMBER OF TRACKED FIBERS
# =============================================================================
# Compares, frame by frame, how many distinct tracks have been created against
# how many fibers the segmentation model actually detects. A large gap between
# the two curves is trajectory fragmentation: one physical fiber represented by
# several shorter tracks.

DIAGNOSTIC_FIBER_COUNT = "800"   # experiment to inspect


def cumulative_tracked_fibers(trajectories):
    """Return (frames, cumulative_track_count) for one trajectory file.

    Entry i of the second list counts the tracks whose first detection occurred
    at a frame <= frames[i].
    """
    first_frame_of_track = {}
    last_frame = 0

    for track_id, track in trajectories.items():
        if track_id == "fibras_por_frame":
            continue
        frames = [f[0] for f in track.get("frame", [])]
        if not frames:
            continue
        first_frame_of_track[track_id] = min(frames)
        last_frame = max(last_frame, max(frames))

    frames_axis = list(range(1, last_frame + 1))
    cumulative = [sum(1 for f in first_frame_of_track.values() if f <= frame)
                  for frame in frames_axis]
    return frames_axis, cumulative


def plot_tracking_summary(raw, filtered):
    """Plot raw tracks, filtered tracks and detections per frame."""
    x_raw, y_raw = cumulative_tracked_fibers(raw)
    x_filtered, y_filtered = cumulative_tracked_fibers(filtered)

    detections = raw.get("fibras_por_frame", [])
    x_detections = list(range(1, len(detections) + 1))

    plt.figure(figsize=(8, 5))
    for x, y, colour, label in (
        (x_raw, y_raw, "tab:blue", "Cumulative tracks (raw)"),
        (x_filtered, y_filtered, "tab:red", "Cumulative tracks (filtered)"),
        (x_detections, detections, "tab:green", "Detections per frame"),
    ):
        if not x:
            continue
        plt.plot(x, y, color=colour, label=label)
        peak = max(y)
        plt.text(x[y.index(peak)], peak, f"max={peak}",
                 color=colour, ha="left", va="bottom", fontsize=9)

    plt.yscale("log")
    plt.xlabel("Frame")
    plt.ylabel("Number of fibers")
    plt.title(f"Tracking summary, {DIAGNOSTIC_FIBER_COUNT}-fiber experiment")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


with open(RESULTS_DIR / f"trajectories_{DIAGNOSTIC_FIBER_COUNT}_raw.json",
          "r", encoding="utf-8") as handle:
    raw_trajectories = json.load(handle)

with open(RESULTS_DIR / f"trajectories_{DIAGNOSTIC_FIBER_COUNT}_filtered.json",
          "r", encoding="utf-8") as handle:
    filtered_trajectories = json.load(handle)

plot_tracking_summary(raw_trajectories, filtered_trajectories)

In [ ]:
# =============================================================================
# DIAGNOSTIC 2: DISTRIBUTION OF THE INSTANTANEOUS VELOCITY COMPONENTS
# =============================================================================
# Velocities are plain finite differences of consecutive centroids, expressed in
# pixels per second. Because the association only links consecutive frames, the
# frames of a track are always contiguous and no gap handling is needed.

def plot_velocity_histograms(trajectories, bin_width=1.0):
    """Histograms of vx and vy, with the modal bin marked on each."""
    vx_samples, vy_samples = [], []

    for track_id, track in trajectories.items():
        if track_id == "fibras_por_frame":
            continue
        centroids = track.get("centroide", [])
        for (x1, y1), (x2, y2) in zip(centroids[:-1], centroids[1:]):
            vx_samples.append((x2 - x1) / DELTA_T)
            vy_samples.append((y2 - y1) / DELTA_T)

    if not vx_samples:
        print("No velocity samples found.")
        return

    figure, axes = plt.subplots(1, 2, figsize=(12, 5))
    figure.suptitle("Instantaneous velocity components")

    for axis, samples, colour, name in (
        (axes[0], vx_samples, "skyblue", "vx"),
        (axes[1], vy_samples, "lightgreen", "vy"),
    ):
        lower = bin_width * math.floor(min(samples) / bin_width)
        upper = bin_width * (math.floor(max(samples) / bin_width) + 1)
        bins = np.arange(lower, upper + bin_width, bin_width)

        counts, edges, _ = axis.hist(samples, bins=bins,
                                     color=colour, edgecolor="black")
        modal_bin = int(np.argmax(counts))
        modal_centre = 0.5 * (edges[modal_bin] + edges[modal_bin + 1])

        axis.axvline(modal_centre, color="red", linestyle="--",
                     label=f"Mode {name} = {modal_centre:.2f} px/s")
        axis.set_title(f"Distribution of {name}")
        axis.set_xlabel(f"{name} (px/s)")
        axis.set_ylabel("Count")
        axis.grid(axis="y", alpha=0.75)
        axis.legend()

    plt.tight_layout()
    plt.show()


plot_velocity_histograms(filtered_trajectories)